In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X,Y
scaler=StandardScaler()
scaler.fit_transform(X)  #data leakage

X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)




In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
X,Y
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=sclaer.tranform(X_test)

clf=LogisticRegression()
clf.fit(X_train_scaled,Y_train)
y_proba=clf.predict_proba(X_test_scaled)

print(average_precision_score(Y_test,y_proba))

In [ ]:
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.pipelines import Pipeline
X,Y


pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('clf',LogisticRegression()),
])

score=cross_val_score(
    pipe,

    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42,scoring='average_precision')
    


In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1000

# Genuine predictive features
hours_studied = np.random.randint(0, 15, n)
attendance = np.random.randint(50, 101, n)
assignments_completed = np.random.randint(0, 11, n)
iq_score = np.random.randint(80, 141, n)
sleep_hours = np.random.randint(4, 10, n)

# Generate target
score = (
    hours_studied * 4
    + attendance * 0.5
    + assignments_completed * 3
    + iq_score * 0.1
)

passed = (score > 90).astype(int)

# LEAKY FEATURE
certificate_issued = passed.copy()

df = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance": attendance,
    "assignments_completed": assignments_completed,
    "iq_score": iq_score,
    "sleep_hours": sleep_hours,
    "certificate_issued": certificate_issued,  # LEAKAGE
    "passed": passed
})

print(df.head())

   hours_studied  attendance  assignments_completed  iq_score  sleep_hours  \
0              6          50                      6       132            8   
1              3         100                      1       136            8   
2             12          94                      3        99            6   
3             14          62                      1        90            9   
4             10          53                      6        95            7   

   certificate_issued  passed  
0                   0       0  
1                   0       0  
2                   1       1  
3                   1       1  
4                   1       1  


In [3]:
import pandas as pd
import numpy as np

def leakage_audit(df, target_col):
    report = []

    print("=" * 70)
    print("LEAKAGE AUDIT REPORT")
    print("=" * 70)

    for col in df.columns:

        if col == target_col:
            continue

        print(f"\nFeature: {col}")

        # Correlation with target
        corr = abs(df[col].corr(df[target_col]))

        # Mean value among positive class
        positive_mean = df[df[target_col] == 1][col].mean()

        # Mean value among negative class
        negative_mean = df[df[target_col] == 0][col].mean()

        diff = abs(positive_mean - negative_mean)

        print(f"Correlation with target: {corr:.4f}")
        print(f"Positive class mean : {positive_mean:.4f}")
        print(f"Negative class mean : {negative_mean:.4f}")
        print(f"Mean difference     : {diff:.4f}")

        suspicion = "LOW"

        if corr > 0.95:
            suspicion = "VERY HIGH"

        elif corr > 0.80:
            suspicion = "HIGH"

        elif corr > 0.60:
            suspicion = "MEDIUM"

        print(f"Leakage suspicion: {suspicion}")

        report.append({
            "feature": col,
            "correlation": round(corr, 4),
            "positive_mean": round(positive_mean, 4),
            "negative_mean": round(negative_mean, 4),
            "mean_difference": round(diff, 4),
            "suspicion": suspicion
        })

    summary = pd.DataFrame(report)

    print("\n")
    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(
        summary.sort_values(
            by="correlation",
            ascending=False
        )
    )

    return summary

summary = leakage_audit(df, "passed")

LEAKAGE AUDIT REPORT

Feature: hours_studied
Correlation with target: 0.7409
Positive class mean : 10.0135
Negative class mean : 3.5667
Mean difference     : 6.4468
Leakage suspicion: MEDIUM

Feature: attendance
Correlation with target: 0.3005
Positive class mean : 79.8923
Negative class mean : 70.9458
Mean difference     : 8.9465
Leakage suspicion: LOW

Feature: assignments_completed
Correlation with target: 0.2840
Positive class mean : 5.6673
Negative class mean : 3.8958
Mean difference     : 1.7715
Leakage suspicion: LOW

Feature: iq_score
Correlation with target: 0.1181
Positive class mean : 112.8500
Negative class mean : 108.6625
Mean difference     : 4.1875
Leakage suspicion: LOW

Feature: sleep_hours
Correlation with target: 0.0181
Positive class mean : 6.4404
Negative class mean : 6.5021
Mean difference     : 0.0617
Leakage suspicion: LOW

Feature: certificate_issued
Correlation with target: 1.0000
Positive class mean : 1.0000
Negative class mean : 0.0000
Mean difference     : 

In [ ]:
import pandas as pd
import numpy as np

# Simulate transaction data with timestamps
np.random.seed(42)
n = 10_000
df = pd.DataFrame({
    'user_id':       np.random.randint(0, 500, n),
    'timestamp':     pd.date_range('2024-01-01', periods=n, freq='10min'),
    'amount':        np.random.exponential(100, n),
    'is_fraud':      np.random.binomial(1, 0.01, n)
})
df = df.sort_values('timestamp').reset_index(drop=True)

# WRONG — computes avg using ALL rows including future ones
df['user_avg_spend_LEAKY'] = df.groupby('user_id')['amount'].transform('mean')

# CORRECT — expanding mean uses only past rows per user
df['user_avg_spend_SAFE'] = (
    df.sort_values('timestamp')
      .groupby('user_id')['amount']
      .transform(lambda x: x.expanding().mean().shift(1))  # shift(1) = no same-row leakage
)

# CORRECT — transaction count in past 24h for each row
def txn_count_past_24h(group):
    group = group.sort_values('timestamp')
    counts = []
    for idx, row in group.iterrows():
        cutoff = row['timestamp'] - pd.Timedelta(hours=24)
        past = group[group['timestamp'] < row['timestamp']]
        past = past[past['timestamp'] >= cutoff]
        counts.append(len(past))
    group['txn_count_24h'] = counts
    return group

df = df.groupby('user_id', group_keys=False).apply(txn_count_past_24h)
print(df[['user_id','timestamp','amount','user_avg_spend_SAFE','txn_count_24h']].head(10))